### Some fun tests


#### Exchange Rate Disparity vs Bridge Traffic

In [ ]:
# Load DKK/SEK monthly exchange rate (how many SEK per 1 DKK)
fx_raw <- read_csv("DKK_SEK Historical Data.csv", show_col_types = FALSE) %>%
  transmute(
    DateMonth = yearmonth(mdy(Date)),
    dkk_sek   = as.numeric(Price)
  ) %>%
  arrange(DateMonth)

# Linear detrend: removes the structural 25-year drift, leaving cyclical disparity
fx_trend_model <- lm(dkk_sek ~ as.numeric(DateMonth), data = fx_raw)

fx_raw <- fx_raw %>%
  mutate(
    fx_trend     = fitted(fx_trend_model),
    cycle        = dkk_sek - fx_trend,          # signed deviation
    disparity    = abs(cycle),                   # point-in-time disparity
    # Rolling averages: exchange rate effects likely accumulate over months,
    # not just react to this month's reading
    disp_roll6   = as.numeric(stats::filter(disparity, rep(1/6,  6),  sides = 1)),
    disp_roll12  = as.numeric(stats::filter(disparity, rep(1/12, 12), sides = 1))
  )

fx_raw %>% summarise(from = min(DateMonth), to = max(DateMonth), n = n())

In [ ]:
# Join FX with traffic for plotting
combined <- traffic_ts %>%
  as_tibble() %>%
  left_join(fx_raw, by = "DateMonth") %>%
  filter(!is.na(dkk_sek))

# Panel 1: raw DKK/SEK with linear trend
p1 <- combined %>%
  ggplot(aes(x = DateMonth)) +
  geom_line(aes(y = dkk_sek), color = "steelblue") +
  geom_line(aes(y = fx_trend), color = "firebrick", linetype = "dashed") +
  labs(
    title = "Fig X: DKK/SEK exchange rate — monthly, Aug 2000–Mar 2026",
    subtitle = "Dashed line: linear trend | Higher = stronger DKK (weaker SEK)",
    x = NULL, y = "SEK per DKK"
  ) +
  theme_minimal()

# Panel 2: disparity (absolute deviation from trend)
p2 <- combined %>%
  ggplot(aes(x = DateMonth, y = disparity)) +
  geom_line(color = "darkorange") +
  geom_hline(yintercept = 0, linetype = "dotted") +
  labs(
    title = "Fig X: Exchange rate disparity — absolute deviation from linear trend",
    subtitle = "Higher = greater divergence from long-run path in either direction",
    x = NULL, y = "|DKK/SEK − trend|"
  ) +
  theme_minimal()

# Panel 3: traffic (for visual comparison of co-movement)
p3 <- combined %>%
  ggplot(aes(x = DateMonth, y = Cars)) +
  geom_line(color = "grey40") +
  labs(
    title = "Fig X: Monthly car crossings — Øresund Bridge",
    x = "Month", y = "Cars"
  ) +
  theme_minimal()

library(patchwork)
p1 / p2 / p3

In [ ]:
# Join FX into one tsibble for CCF tests
combined_ts <- traffic_ts %>%
  left_join(fx_raw %>% select(DateMonth, disparity, disp_roll6, disp_roll12), 
            by = "DateMonth") %>%
  filter(!is.na(disp_roll12))  # trim rows where rolling window isn't yet full

# ── Test 1: CCF against ARIMA residuals (lag 0-24) ───────────────────────────
# Point-in-time disparity vs ARIMA residuals
arima_resid <- fit %>%
  select(arima212011) %>%
  residuals() %>%
  as_tibble() %>%
  select(DateMonth, .resid)

ccf_arima <- arima_resid %>%
  left_join(combined_ts %>% as_tibble() %>% 
              select(DateMonth, disparity, disp_roll12), by = "DateMonth") %>%
  filter(!is.na(disp_roll12)) %>%
  as_tsibble(index = DateMonth)

ccf_arima %>%
  CCF(.resid, disparity, lag_max = 24) %>%
  autoplot() +
  labs(title = "Fig X: CCF — ARIMA residuals vs point-in-time disparity (lags 0–24)",
       subtitle = "AR/MA components already absorb autocorrelation — remaining signal only",
       x = "Lag (months)", y = "Cross-correlation") +
  theme_minimal()

# 12-month rolling disparity — tests whether sustained divergence matters more
ccf_arima %>%
  CCF(.resid, disp_roll12, lag_max = 24) %>%
  autoplot() +
  labs(title = "Fig X: CCF — ARIMA residuals vs 12-month rolling disparity (lags 0–24)",
       subtitle = "Rolling average captures cumulative exchange rate pressure",
       x = "Lag (months)", y = "Cross-correlation") +
  theme_minimal()

# ── Test 2: TSLM residuals — no AR/MA absorbing variance ─────────────────────
# Strip back to trend + season + covid only; exchange rate gets uncontested variance
fit_tslm <- combined_ts %>%
  model(TSLM(box_cox(Cars, lambda) ~ trend() + season() + covid))

tslm_resid <- fit_tslm %>%
  residuals() %>%
  as_tibble() %>%
  select(DateMonth, .resid)

ccf_tslm <- tslm_resid %>%
  left_join(combined_ts %>% as_tibble() %>% 
              select(DateMonth, disparity, disp_roll12), by = "DateMonth") %>%
  filter(!is.na(disp_roll12)) %>%
  as_tsibble(index = DateMonth)

ccf_tslm %>%
  CCF(.resid, disparity, lag_max = 24) %>%
  autoplot() +
  labs(title = "Fig X: CCF — TSLM residuals vs point-in-time disparity (lags 0–24)",
       subtitle = "TSLM has no AR/MA terms — exchange rate competes against trend + season only",
       x = "Lag (months)", y = "Cross-correlation") +
  theme_minimal()

ccf_tslm %>%
  CCF(.resid, disp_roll12, lag_max = 24) %>%
  autoplot() +
  labs(title = "Fig X: CCF — TSLM residuals vs 12-month rolling disparity (lags 0–24)",
       subtitle = "Rolling disparity vs unexplained variation after trend and season",
       x = "Lag (months)", y = "Cross-correlation") +
  theme_minimal()

In [ ]:
# Direct TSLM test — exchange rate as explicit regressor with no AR/MA competition
# If significant here, the effect is genuine and not an artefact of ARMA specification
fit_tslm_fx <- combined_ts %>%
  model(
    tslm_base    = TSLM(box_cox(Cars, lambda) ~ trend() + season() + covid),
    tslm_disp    = TSLM(box_cox(Cars, lambda) ~ trend() + season() + covid + disparity),
    tslm_roll6   = TSLM(box_cox(Cars, lambda) ~ trend() + season() + covid + disp_roll6),
    tslm_roll12  = TSLM(box_cox(Cars, lambda) ~ trend() + season() + covid + disp_roll12)
  )

# AIC comparison
glance(fit_tslm_fx) %>% select(.model, AIC, AICc, BIC) %>% arrange(AICc)

# Coefficient details for the best model
fit_tslm_fx %>% select(tslm_roll12) %>% report()

# Also keep the ARIMA dynamic regression for comparison
glance(fit_tslm_fx) %>% select(.model, AIC, AICc, BIC) %>% arrange(AICc)

In [ ]:
# Robustness check: does the result survive excluding 2008-2012?
# The SEK crashed hard during the financial crisis — if that single episode
# is driving the correlation, the coefficient will collapse here
combined_ex_crisis <- combined_ts %>%
  filter(DateMonth < yearmonth("2008 Jan") | DateMonth > yearmonth("2012 Dec"))

fit_robust <- combined_ex_crisis %>%
  model(
    base     = TSLM(box_cox(Cars, lambda) ~ trend() + season() + covid),
    roll12   = TSLM(box_cox(Cars, lambda) ~ trend() + season() + covid + disp_roll12)
  )

glance(fit_robust) %>% select(.model, AIC, AICc, BIC) %>% arrange(AICc)

# Coefficient on disp_roll12 outside the crisis period
fit_robust %>% select(roll12) %>% tidy() %>%
  filter(term == "disp_roll12") %>%
  select(term, estimate, std.error, statistic, p.value)

#### Analysis

**The exchange rate hypothesis does not hold up under scrutiny.**

---

**Initial result (full sample)**

With the full 2000–2026 sample and no AR/MA competition, the 12-month rolling disparity appeared significant: coefficient = +22,374, p = 0.003, AIC improvement of 7.4 units over the base TSLM. Directionally correct and statistically clean.

**Robustness check: exclude 2008–2012**

Removing the financial crisis period collapses the result entirely:

| | Full sample | Ex 2008–2012 |
|---|---|---|
| AIC improvement | 7.4 units | 0.49 units |
| Coefficient | 22,374 | 12,848 |
| p-value | 0.003 | 0.127 |

The AIC improvement falls to 0.49 — smaller than the 2-unit parameter penalty, meaning the variable no longer earns its place in the model. The coefficient halves and loses significance. The full-sample result was almost entirely driven by a single 5-year window.

**Why 2008–2012 drove the result**

During the financial crisis, the SEK weakened sharply relative to the DKK, pushing DKK/SEK well above its linear trend and generating large disparity values. Simultaneously, Øresund traffic continued to grow — the corridor's structural economic integration was strong enough that the mild 2009 slowdown barely registered in the residuals (unlike COVID, the financial crisis produced no outlier z-scores in the STL remainder). These two facts coinciding — extreme disparity and continued traffic growth — created a strong apparent correlation. But they are both symptoms of the same underlying shock (relative macroeconomic divergence between Sweden and Denmark during the crisis), not a causal chain running from exchange rates to crossing decisions.

**Conclusion**

At normal levels of exchange rate disparity, there is no detectable effect on Øresund Bridge traffic. The earlier p = 0.003 result should not be taken at face value. This is itself an informative finding: the bridge's traffic is driven by structural economic integration between the two cities — employment, population, and commuter patterns — and is insensitive to the short- to medium-run currency fluctuations that the exchange rate captures. Establishing a genuine exchange rate effect would require directional crossing data (Denmark→Sweden vs Sweden→Denmark separately) and a control for the relative economic output gap between the two countries to separate currency effects from broader economic divergence.

#### Growth Rate Deceleration

In [ ]:
# Year-over-year growth rate: Cars_t / Cars_{t-12} - 1
# Excludes the first 12 months (no prior year to compare against)
growth_ts <- traffic_ts %>%
  mutate(
    yoy = (Cars / lag(Cars, 12) - 1) * 100
  ) %>%
  filter(!is.na(yoy))

growth_ts %>%
  ggplot(aes(x = DateMonth, y = yoy)) +
  geom_hline(yintercept = 0, linetype = "dashed", color = "grey60") +
  geom_line(color = "steelblue", linewidth = 0.4) +
  geom_smooth(method = "loess", span = 0.35, se = TRUE,
              color = "firebrick", fill = "firebrick", alpha = 0.15) +
  annotate("rect",
           xmin = yearmonth("2020 Mar"), xmax = yearmonth("2021 Jul"),
           ymin = -Inf, ymax = Inf,
           fill = "orange", alpha = 0.15) +
  annotate("text", x = yearmonth("2020 Sep"), y = 55,
           label = "COVID", size = 3, color = "darkorange") +
  labs(
    title = "Fig X: Year-over-year growth rate — Øresund Bridge car traffic",
    subtitle = "Blue: monthly YoY % | Red: LOESS trend | Shaded: COVID window",
    x = "Month", y = "YoY growth (%)"
  ) +
  theme_minimal()

In [ ]:
# Test for structural deceleration in growth rates
# Exclude COVID window — those observations are outliers not representative of trend
growth_clean <- growth_ts %>%
  as_tibble() %>%
  filter(covid == 0) %>%
  mutate(t = as.numeric(DateMonth))

# Linear trend on growth rates: negative slope = deceleration
lm_decel <- lm(yoy ~ t, data = growth_clean)
summary(lm_decel)

# Decade-by-decade average growth — is each decade slower than the last?
growth_clean %>%
  mutate(decade = case_when(
    DateMonth < yearmonth("2010 Jan") ~ "2000s",
    DateMonth < yearmonth("2020 Jan") ~ "2010s",
    TRUE                              ~ "2020s"
  )) %>%
  group_by(decade) %>%
  summarise(
    mean_yoy  = round(mean(yoy), 2),
    sd_yoy    = round(sd(yoy), 2),
    n_months  = n(),
    .groups   = "drop"
  )

In [ ]:
# Visual: growth rate distribution shifting over decades
growth_clean %>%
  mutate(decade = case_when(
    DateMonth < yearmonth("2010 Jan") ~ "2000s",
    DateMonth < yearmonth("2020 Jan") ~ "2010s",
    TRUE                              ~ "2020s"
  )) %>%
  ggplot(aes(x = yoy, fill = decade)) +
  geom_density(alpha = 0.5) +
  geom_vline(xintercept = 0, linetype = "dashed", color = "grey40") +
  scale_fill_manual(values = c("2000s" = "steelblue",
                               "2010s" = "firebrick",
                               "2020s" = "darkgreen")) +
  labs(
    title = "Fig X: Distribution of YoY growth rates by decade",
    subtitle = "Rightward shift = faster growth era | Leftward shift = slowdown",
    x = "YoY growth (%)", y = "Density", fill = "Decade"
  ) +
  theme_minimal()

In [ ]:
# The 2020s decade average is distorted: post-COVID rebound months (Aug 2021–Jul 2022)
# have covid=0 but compare against depressed base months — not real growth
# Isolate the genuine post-rebound period: Aug 2022 onward
growth_ts %>%
  as_tibble() %>%
  filter(covid == 0) %>%
  mutate(period = case_when(
    DateMonth < yearmonth("2010 Jan")  ~ "2000s",
    DateMonth < yearmonth("2020 Jan")  ~ "2010s",
    DateMonth < yearmonth("2022 Aug")  ~ "2020–22 rebound",  # inflated base effect
    TRUE                               ~ "2022+ genuine"
  )) %>%
  group_by(period) %>%
  summarise(
    mean_yoy = round(mean(yoy), 2),
    sd_yoy   = round(sd(yoy), 2),
    n_months = n(),
    .groups  = "drop"
  )